In [ ]:
!pip install pdf2image

In [ ]:
!brew install poppler

In [ ]:
!which pdfinfo

In [ ]:
import os
from pathlib import Path
import pandas as pd
import re
from tqdm import tqdm
import json
from pdf2image import convert_from_path

In [ ]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(
    use_angle_cls=True,  # rotation
    lang="fr"            # OCR français
)  # pas de show_log
print("PaddleOCR initialized successfully.")


## nouvelle installation

In [ ]:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

In [ ]:
!python -m pip install "paddleocr[all]"

## initialisation OCR

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False, 
    use_doc_unwarping=False, 
    use_textline_orientation=False) # text detection + text recognition

## gérer les colonnes 

In [ ]:
from paddleocr import PPStructureV3

In [ ]:
layout = PPStructureV3(
    lang="fr",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False
)

In [ ]:
path='./data_test/IIIF_1000w.jpg'

output = layout.predict(input=path)
for res in output:
    res.save_to_json(save_path="output")
    print('json correspondant enregistré')

In [ ]:
list_images = []

for path in list_images : 
    output = layout.predict(input=path)
    print('1 ocr de fait')
    for res in output:
        res.save_to_json(save_path="output")
        print('json correspondant enregistré')

### pipeline complète (reverser vers pdf2image.py par la suite)   : 

In [ ]:
import cv2

def extract_text_blocks(img_path):
    result = layout(img_path)
    blocks = []
    for b in result:
        if b.get("type") != "text":
            continue
        x0, y0, x1, y1 = b["bbox"]
        # b["res"] est souvent une liste de lignes OCR
        text_lines = []
        if isinstance(b.get("res"), list):
            for r in b["res"]:
                t = r.get("text", "")
                if t:
                    text_lines.append(t)
        elif isinstance(b.get("res"), dict):
            t = b["res"].get("text", "")
            if t:
                text_lines.append(t)
        blocks.append({
            "bbox": (x0, y0, x1, y1),
            "text": "\n".join(text_lines).strip(),
        })
    return blocks

def order_blocks(blocks, page_w):
    # sépare blocs pleine largeur vs colonnes
    full = []
    left = []
    right = []
    mid = page_w / 2

    for b in blocks:
        x0, y0, x1, y1 = b["bbox"]
        w = x1 - x0
        if w >= 0.6 * page_w:
            full.append(b)
        else:
            if (x0 + x1) / 2 < mid:
                left.append(b)
            else:
                right.append(b)

    # si pas vraiment deux colonnes, ordre simple
    if len(left) < 2 or len(right) < 2:
        return sorted(blocks, key=lambda b: (b["bbox"][1], b["bbox"][0]))

    full.sort(key=lambda b: b["bbox"][1])

    # segmente l’espace vertical entre blocs pleine largeur
    segments = []
    prev_y = -1
    for fb in full:
        segments.append((prev_y, fb["bbox"][1]))
        prev_y = fb["bbox"][3]
    segments.append((prev_y, float("inf")))

    def in_segment(b, seg):
        y0 = b["bbox"][1]
        return seg[0] < y0 <= seg[1]

    ordered = []
    for i, seg in enumerate(segments):
        l = [b for b in left if in_segment(b, seg)]
        r = [b for b in right if in_segment(b, seg)]
        l.sort(key=lambda b: b["bbox"][1])
        r.sort(key=lambda b: b["bbox"][1])
        ordered.extend(l)
        ordered.extend(r)
        if i < len(full):
            ordered.append(full[i])
    return ordered

def ocr_page_with_layout(img_path):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    blocks = extract_text_blocks(img_path)
    ordered = order_blocks(blocks, w)
    text = "\n\n".join([b["text"] for b in ordered if b["text"]])
    return text


## appliquer l'océrisation sur blocs de texte

In [ ]:
#test 1 colonne simple
result = ocr.predict("./test_revue_fr.jpeg")
for res in result:
    res.print()
    res.save_to_json("output")

In [ ]:
#test 3 colonnes
result = ocr.predict("./plusieurs_colonnes.jpeg")
for res in result:
    res.print()
    res.save_to_json("output")

In [ ]:
#test 1 colonne puis 2 
result = ocr.predict("./revue_scientifique_from_web.jpeg")
for res in result:
    res.print()
    res.save_to_json("output")

## de pdf à image à partir dossier des pdfs vers dossier où chaque document est un répertoire de jpg ordonné

### ouvrir les fichiers

In [ ]:
path_pdf = "/Users/mathieu/Documents/memoire/test_ocr/"

In [ ]:
from glob import glob
files_name = []
for file in glob(path_pdf+"*"):
    print(file)
    files_name.append(file)

## De pdf à jpegs

In [ ]:
import glob
for file in tqdm(files_name) :
    path = file+"/*.pdf"
    pdf = glob(path) 
    print(pdf)

    convert_from_path(pdf[0],
        dpi=200,
        output_folder=file, 
        poppler_path="/opt/homebrew/bin",
        fmt="jpeg",
        output_file="page")
    
    

## PaddleOCR sur les jpeg --> json

In [ ]:
#version à plusieurs jsonls

import glob
import os
import json
from tqdm.auto import tqdm

folder = "/Users/mathieu/Documents/memoire/test_ocr/revue_scientifique18840701"
paths_images = sorted(glob.glob(folder + "/*.jpg"))

# il faut que ocr soit défini avant
for idx, img_path in enumerate(tqdm(paths_images, total=len(paths_images)), start=1):
    print('la boucle commence')
    res = ocr.predict(img_path)
    print("l'ocr a fonctionné")
    base = os.path.splitext(os.path.basename(img_path))[0]
    out_path = os.path.join(folder, f"{base}.jsonl")
    print("il ne reste plus qu'à mettre le texte dans le json")
    with open(out_path, "w") as f:
        f.write(json.dumps(
            {"page": idx, "image": os.path.basename(img_path), "ocr": res},
            ensure_ascii=False
        ) + "\n")

    print('json enregistré')


In [ ]:
#ouvrir mes différents jsonl et en faire un seul json
jsonl_files = glob.glob("/Users/mathieu/Documents/memoire/test_ocr/*.jsonl")
records = []
for path in jsonl_files:
    with open(path, "r") as f:
        for line in f:
            records.append(json.loads(line))

with open("merged_ocr.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Fusionné {len(records)} entrées depuis {len(jsonl_files)} fichiers.")

#supprimer les jpg dans mon répertoire et les jsonls